# 🧠 Analisis Dampak Media Sosial terhadap Kesehatan Mental Remaja
**Metode: CRISP-DM | Algoritma: Decision Tree Classifier**

---

> Dataset: [Teen Mental Health Dataset — Kaggle](https://www.kaggle.com/datasets/sunil123kumar/social-media-impact-on-mental-health)

Notebook ini mengikuti kerangka kerja **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*) yang terdiri dari 6 fase:

| # | Fase | Deskripsi |
|---|------|-----------|
| 1 | Business Understanding | Memahami tujuan dan rumusan masalah |
| 2 | Data Understanding | Eksplorasi awal dataset |
| 3 | Data Preparation | Pembersihan & transformasi data |
| 4 | Modeling | Membangun model Decision Tree |
| 5 | Evaluation | Mengukur performa model |
| 6 | Deployment | Interpretasi & kesimpulan |


---
## ⚙️ Setup & Instalasi Library

In [ ]:
import kagglehub
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ Semua library berhasil diimport.")

---
## 📌 Fase 1 — Business Understanding

### Latar Belakang
Penggunaan media sosial yang berlebihan di kalangan remaja telah menjadi perhatian serius dalam dunia kesehatan mental. Penelitian ini bertujuan untuk mengidentifikasi apakah pola penggunaan media sosial dapat menjadi prediktor **risiko depresi** pada remaja.

### Rumusan Masalah
> **Apakah kebiasaan penggunaan media sosial (durasi, platform, interaksi sosial) dapat digunakan untuk memprediksi label depresi pada remaja?**

### Tujuan
- Menganalisis korelasi antara kebiasaan media sosial dengan tingkat depresi.
- Membangun model klasifikasi **Decision Tree** untuk memprediksi `depression_label`.
- Mengidentifikasi fitur-fitur yang paling berpengaruh terhadap depresi.

### Target
- **Target variabel**: `depression_label` (0 = Tidak Depresi, 1 = Depresi)
- **Metrik sukses**: Akurasi model ≥ 75%

---
## 📂 Fase 2 — Data Understanding

### 2.1 Load Dataset

In [ ]:
# Download dataset dari Kaggle
path = kagglehub.dataset_download("sunil123kumar/social-media-impact-on-mental-health")
print("📁 Path dataset:", path)

# Load ke DataFrame
df = pd.read_csv(f"{path}/Teen_Mental_Health_Dataset.csv")
print(f"✅ Dataset berhasil dimuat — {df.shape[0]} baris, {df.shape[1]} kolom")

### 2.2 Eksplorasi Awal Data

In [ ]:
# Tampilkan 5 baris pertama
print("=== Preview Dataset ===")
df.head()

In [ ]:
# Informasi tipe data dan missing values
print("=== Info Dataset ===")
df.info()

In [ ]:
# Statistik deskriptif
print("=== Statistik Deskriptif ===")
df.describe()

In [ ]:
# Cek missing values
print("=== Missing Values per Kolom ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ Tidak ada missing values.")

# Distribusi target
print("\n=== Distribusi Target (depression_label) ===")
print(df['depression_label'].value_counts())
print(f"Proporsi: {df['depression_label'].value_counts(normalize=True).round(2).to_dict()}")

### 2.3 Analisis Korelasi Antar Fitur Numerik

In [ ]:
# Pilih kolom numerik
df_numeric = df.select_dtypes(include=['int64', 'float64'])
corr_matrix = df_numeric.corr()

# Heatmap korelasi lengkap
plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)
plt.title('Heatmap Korelasi Antar Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi khusus terhadap variabel target
corr_depression = corr_matrix[['depression_label']].sort_values(
    by='depression_label', ascending=False
)

plt.figure(figsize=(6, 7))
sns.heatmap(
    corr_depression,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)
plt.title('Korelasi Fitur terhadap Depression Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Interpretasi:")
print("Tiga fitur dengan korelasi tertinggi terhadap depresi:")
print("  1. daily_social_media_hours — durasi penggunaan media sosial per hari")
print("  2. stress_level             — tingkat stres yang dialami remaja")
print("  3. anxiety_level            — tingkat kecemasan yang dialami remaja")

---
## 🔧 Fase 3 — Data Preparation

Pada fase ini kami melakukan:
1. Pemisahan fitur (`X`) dan target (`y`)
2. Encoding kolom kategorikal menggunakan `LabelEncoder`
3. Pembagian data menjadi data latih dan data uji (80:20)

In [ ]:
# Langkah 1 — Pisahkan Fitur dan Target
X = df.drop('depression_label', axis=1).copy()
y = df['depression_label']

print(f"✅ Fitur (X): {X.shape} | Target (y): {y.shape}")
print(f"\nKolom fitur: {list(X.columns)}")

In [ ]:
# Langkah 2 — Encoding Kolom Kategorikal
le_gender   = LabelEncoder()
le_platform = LabelEncoder()
le_social   = LabelEncoder()

X['gender']                  = le_gender.fit_transform(X['gender'])
X['platform_usage']          = le_platform.fit_transform(X['platform_usage'])
X['social_interaction_level']= le_social.fit_transform(X['social_interaction_level'])

print("✅ Encoding selesai. Semua fitur kini bertipe numerik.")
X.info()

In [ ]:
# Langkah 3 — Split Data Latih dan Data Uji (80:20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # Menjaga proporsi kelas tetap seimbang
)

print(f"✅ Data Latih : {X_train.shape[0]} sampel")
print(f"   Data Uji  : {X_test.shape[0]} sampel")
print(f"\nDistribusi kelas pada data latih:\n{y_train.value_counts()}")

---
## 🤖 Fase 4 — Modeling

Kami menggunakan algoritma **Decision Tree Classifier** dengan konfigurasi:
- `criterion='gini'` — menggunakan Gini Impurity sebagai kriteria pemisahan
- `max_depth=4` — membatasi kedalaman pohon agar tidak overfitting

In [ ]:
# Inisialisasi dan latih model
model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)
print("✅ Model Decision Tree berhasil dilatih.")
print(f"   Kedalaman pohon aktual: {model.get_depth()}")
print(f"   Jumlah leaf node      : {model.get_n_leaves()}")

In [ ]:
# Visualisasi Struktur Pohon Keputusan
plt.figure(figsize=(22, 10))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=['No Depression', 'Depression'],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Struktur Pohon Keputusan (Decision Tree)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📊 Fase 5 — Evaluation

Kami mengevaluasi performa model menggunakan:
- **Accuracy Score** — persentase prediksi yang benar
- **Classification Report** — precision, recall, f1-score per kelas
- **Confusion Matrix** — visualisasi kesalahan klasifikasi

In [ ]:
# Prediksi pada data uji
y_pred = model.predict(X_test)

# Akurasi
acc = accuracy_score(y_test, y_pred)
print(f"🎯 Accuracy Score : {acc:.4f} ({acc*100:.2f}%)")

In [ ]:
# Classification Report
print("=== Classification Report ===")
print(classification_report(
    y_test, y_pred,
    target_names=['No Depression (0)', 'Depression (1)']
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Depression', 'Depression'],
    yticklabels=['No Depression', 'Depression']
)
plt.xlabel('Predicted Label', fontweight='bold')
plt.ylabel('Actual Label', fontweight='bold')
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negative  (TN): {tn}  — Benar diprediksi Tidak Depresi")
print(f"False Positive (FP): {fp}  — Salah diprediksi Depresi")
print(f"False Negative (FN): {fn}  — Salah diprediksi Tidak Depresi")
print(f"True Positive  (TP): {tp}  — Benar diprediksi Depresi")

In [ ]:
# Feature Importance
importance = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance,
    x='Importance',
    y='Feature',
    palette='viridis'
)
plt.title('Feature Importance — Decision Tree', fontsize=13, fontweight='bold')
plt.xlabel('Tingkat Kepentingan')
plt.ylabel('Fitur')
plt.tight_layout()
plt.show()

print("\n📋 Ranking Feature Importance:")
print(importance.to_string(index=False))

---
## 🚀 Fase 6 — Deployment (Interpretasi & Kesimpulan)

### Hasil Evaluasi Model

Model Decision Tree yang kami bangun berhasil mengklasifikasikan risiko depresi remaja berdasarkan pola penggunaan media sosial.

### Temuan Utama

1. **Fitur Paling Berpengaruh**: Berdasarkan `feature_importances_`, fitur seperti `stress_level`, `daily_social_media_hours`, dan `anxiety_level` memiliki kontribusi terbesar dalam keputusan model.

2. **Korelasi Positif**: Semakin lama remaja menggunakan media sosial per hari, semakin tinggi potensi risiko depresi yang dimiliki.

3. **Performa Model**: Model mencapai akurasi yang cukup baik pada data uji dengan kedalaman pohon yang dibatasi (max_depth=4) untuk mencegah overfitting.

### Rekomendasi

| Stakeholder | Rekomendasi |
|-------------|-------------|
| Orang Tua | Memantau durasi penggunaan media sosial anak |
| Sekolah | Mengadakan program literasi digital dan kesehatan mental |
| Remaja | Membatasi screen time dan meningkatkan interaksi sosial langsung |
| Peneliti | Mengeksplorasi model lain (Random Forest, XGBoost) untuk performa lebih baik |

### Keterbatasan
- Dataset bersifat sintetis/simulasi, sehingga perlu validasi lebih lanjut dengan data nyata.
- Model belum diuji pada distribusi data yang berbeda (cross-validation).
- Faktor lain di luar media sosial (keluarga, ekonomi) belum dimasukkan ke dalam model.